In [ ]:
import os
from langchain_community.chat_models import ChatZhipuAI
from langchain_core.messages import HumanMessage, SystemMessage
import httpx

In [ ]:
http_client = httpx.Client(timeout=httpx.Timeout(120.0, connect=10.0))

os.environ['ZHIPUAI_API_KEY'] = '838d6707ed7142408fc2d529e94ad9d2.HQs5sQJCTIbndQu8'
llm = ChatZhipuAI(
    model='glm-4.7-flash',
    temperature=0.2,
    http_client='http_client'

)

# response = llm.invoke('你好, 请介绍你自己')
# print(response.content)

In [ ]:
messages = [
    SystemMessage(content='你是一个python专家, 用简洁的语言解释问题'),
    HumanMessage(content='什么是闭包')
]
# response = llm.invoke(messages)
# print(response.content)
for chunk in llm.stream(messages):
    print(chunk.content, end='', flush=True)

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("system", "你是一个{role}，用{style}的风格回答问题"),
    ("human", "{question}")
])

formatted_prompt = prompt.format_messages(
    role='python老师',
    style='简洁明了',
    question='解释列表推导式'
)

for chunk in llm.stream(formatted_prompt):
    print(chunk.content, end='', flush=True)

In [ ]:
import os
from langchain_community.chat_models import ChatZhipuAI
from langchain_community.embeddings import ZhipuAIEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.documents import Document

# 设置API Key
os.environ["ZHIPUAI_API_KEY"] = '838d6707ed7142408fc2d529e94ad9d2.HQs5sQJCTIbndQu8'

# 1. 准备简单的文本数据
simple_texts = [
    """鲁滨逊在荒岛上生活了28年。他从沉船上搬来了食物、枪支、工具，在岛上搭建帐篷，种植大麦，驯养山羊。""",
    
    """星期五是鲁滨逊救下的野人。那天星期五，一群野人带着俘虏上岛准备吃掉，鲁滨逊救下了其中一个，取名"星期五"。""",
    
    """鲁滨逊出生于英国约克城一个商人家庭，从小就渴望航海。他瞒着父母出海，经历了多次冒险，最后流落荒岛。""",
    
    """在岛上，鲁滨逊遇到了很多困难：没有食物、没有住所、害怕野兽和野人。但他通过自己的努力一一克服了。""",
    
    """鲁滨逊救下星期五后，教他说话、种植、做面包。星期五成为鲁滨逊忠实的仆人和朋友。"""
]

# 2. 创建文档对象
documents = []
for i, text in enumerate(simple_texts):
    doc = Document(
        page_content=text,
        metadata={"source": f"片段{i+1}"}
    )
    documents.append(doc)

print(f"准备了 {len(documents)} 个文档片段")

# 3. 创建向量数据库
print("创建向量数据库...")
embeddings = ZhipuAIEmbeddings(model="embedding-2")
vectorstore = Chroma.from_documents(
    documents=documents,
    embedding=embeddings
)

# 4. 创建检索器
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

# 5. 创建提示模板
prompt = ChatPromptTemplate.from_messages([
    ("system", "基于以下信息回答问题。如果信息不够，就说不知道。\n\n{context}"),
    ("human", "{question}")
])

# 6. 创建LLM
llm = ChatZhipuAI(model="glm-4.7", temperature=0.3)

# 7. RAG查询函数
def rag_query(question):
    print(f"\n问题: {question}")
    
    # 检索相关文档
    docs = retriever.get_relevant_documents(question)
    print(f"找到 {len(docs)} 个相关片段")
    
    # 构建上下文
    context = "\n\n".join([doc.page_content for doc in docs])
    print(f"上下文: {context[:100]}...")
    
    # 生成回答
    messages = prompt.format_messages(
        context=context,
        question=question
    )
    response = llm.invoke(messages)
    
    print(f"回答: {response.content}")
    print("-" * 50)
    
    return response.content

# 8. 测试
if __name__ == "__main__":
    # 测试几个问题
    questions = [

        '鲁滨逊哪一年回到了家'
    ]
    
    for q in questions:
        rag_query(q)

    # # 9. 交互模式
    # print("\n" + "="*50)
    # print("简单问答模式 (输入quit退出)")
    # print("="*50)
    
    # while True:
    #     q = input("\n请输入问题: ").strip()
    #     if q.lower() == 'quit':
    #         break
    #     if q:
    #         rag_query(q)

In [ ]:
import os
from langchain_community.chat_models import ChatZhipuAI
from langchain_community.embeddings import ZhipuAIEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.documents import Document

In [ ]:
os.environ['ZHIPU_API_KEY'] = '838d6707ed7142408fc2d529e94ad9d2.HQs5sQJCTIbndQu8'

simple_texts = [
    """鲁滨逊在荒岛上生活了28年。他从沉船上搬来了食物、枪支、工具，在岛上搭建帐篷，种植大麦，驯养山羊。""",
    
    """星期五是鲁滨逊救下的野人。那天星期五，一群野人带着俘虏上岛准备吃掉，鲁滨逊救下了其中一个，取名"星期五"。""",
    
    """鲁滨逊出生于英国约克城一个商人家庭，从小就渴望航海。他瞒着父母出海，经历了多次冒险，最后流落荒岛。""",
    
    """在岛上，鲁滨逊遇到了很多困难：没有食物、没有住所、害怕野兽和野人。但他通过自己的努力一一克服了。""",
    
    """鲁滨逊救下星期五后，教他说话、种植、做面包。星期五成为鲁滨逊忠实的仆人和朋友。"""
]

documents = []

for i, text in enumerate(simple_texts):
    doc = Document(
        page_content=text,
        metadata={'source': f'片段{i+1}'}
    )
    documents.append(doc)

embeddings = ZhipuAIEmbeddings(model='embedding-2')
vectorstore = Chroma.from_documents(
    documents=documents,
    embedding=embeddings
)

retriever = vectorstore.as_retriever(search_kwargs={'k': 2})

prompt = ChatPromptTemplate.from_messages([
    ('system', '基于以下信息回答问题, 如果信息不够就说不知道'),
    ('human', '{question}')
])

llm = ChatZhipuAI(model='glm-4.7', temperature=0.2)

def rag_query(question):
    print(f'\n问题: {question}')
    docs = retriever.get_relevant_documents(question)
    print(f'找到 {len(docs)}个相关片段')

    context = '\n\n'.join([doc.page_content for doc in docs])

    messages = prompt.format_messages(
        context=context,
        question=question
    )
    response = llm.invoke(messages)
    print(response.content)

question = '鲁滨逊在荒岛上生活了多少年'
rag_query(question)

In [1]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_community.chat_models import ChatZhipuAI
import os

os.environ['ZHIPUAI_API_KEY'] = '838d6707ed7142408fc2d529e94ad9d2.HQs5sQJCTIbndQu8'

d:\anaconda3\envs\tf2\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
llm = ChatZhipuAI(model='glm-4.7')
prompt = ChatPromptTemplate.from_messages([
    ('system', '你是一名算法专家'),
    ('user', '{input}')
])

chain = prompt | llm
result = chain.invoke({'input': '写一篇AI的技术文章, 100字左右'})
print(result)

content='人工智能正经历从感知智能到认知智能的跃迁。基于Transformer架构的大语言模型，通过海量数据预训练与微调，展现出惊人的涌现能力。强大的算法与算力协同，让机器具备复杂的逻辑推理与创造力。尽管面临可解释性等挑战，但迈向通用人工智能（AGI）的步伐已不可逆转。' additional_kwargs={} response_metadata={'token_usage': {'completion_tokens': 706, 'completion_tokens_details': {'reasoning_tokens': 630}, 'prompt_tokens': 20, 'prompt_tokens_details': {'cached_tokens': 3}, 'total_tokens': 726}, 'model_name': 'glm-4.7', 'finish_reason': 'stop'} id='run--019cd83b-a28b-7c30-8195-b08c3654729e-0'
